In [ ]:
import edgar as et, os, bs4
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
et.set_identity(os.getenv("EDGAR_IDENTITY"))
apple = et.Company("AAPL")
#get 1 10-q fillings for apple
filings = apple.get_filings(form="10-Q")
filing = filings.latest()

'**UNITED STATES**\n\n**SECURITIES AND EXCHANGE COMMISSION**\n\n### Washington, D.C. 20549\n\n### FORM  10-Q\n\n### (Mark One)\n\n☒\n\n### QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\n### For the quarterly period ended  June 27, 2026\n\n### or\n\n☐\n\n### TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the transition period from   to  \\.\n\n### Commission File Number:\n\n**001\\-36743**\n\n![g66145g66i43.jpg](https://www.sec.gov/Archives/edgar/data/320193/000032019326000020/aapl-20260627_g1.jpg)\n\n**Apple Inc\\.**\n\n\\(Exact name of Registrant as specified in its charter\\)\n\n| California | 94-2404110 |\n| (State or other jurisdiction\n\n\nof incorporation or organization) | (I.R.S. Employer Identification No.) |\n| One Apple Park Way |  |\n| Cupertino, California | 95014 |\n| (Address of principal executive offices) | (Zip Code) |\n\n**\\( 408 \\)  996\\-1010**\n\n\\(Registrant’s telephon

In [40]:
apple.tickers[0]

'AAPL'

In [13]:
text = filing.text()
text[:5000]

'UNITED STATES\n\nSECURITIES AND EXCHANGE COMMISSION\n\nWashington, D.C. 20549\n\nFORM 10-Q\n\n(Mark One)\n\n☒ QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the quarterly period ended June 27, 2026\n\nor\n\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n\nFor the transition period from to.\n\nCommission File Number: 001-36743\n\nApple Inc.\n\n(Exact name of Registrant as specified in its charter)\n\n  California                                    94-2404110\n  (State or other jurisdiction                  (I.R.S. Employer Identification No.)\n\n\n  of incorporation or organization)\n  One Apple Park Way\n  Cupertino, California                         95014\n  (Address of principal executive offices)      (Zip Code)\n\n( 408 ) 996-1010\n\n(Registrant’s telephone number, including area code)\n\nSecurities registered pursuant to Section 12(b) of the Act:\n\n  Title of each class                 

In [27]:
tenq = filing.obj()

print(type(tenq))
dir(tenq)[len(dir(tenq))-24:len(dir(tenq)) - 1]


<class 'edgar.company_reports.ten_q.TenQ'>


['auditor',
 'balance_sheet',
 'cash_flow_statement',
 'chunked_document',
 'company',
 'doc',
 'document',
 'filing_date',
 'financials',
 'form',
 'get',
 'get_item_with_part',
 'get_structure',
 'grep',
 'income_statement',
 'items',
 'notes',
 'period_of_report',
 'reports',
 'sections',
 'signatures',
 'structure',
 'to_context']

In [34]:
print(tenq.items)
tenq.get_structure()

['Part I, Item 1', 'Part I, Item 2', 'Part I, Item 3', 'Part I, Item 4', 'Part II, Item 1', 'Part II, Item 1A', 'Part II, Item 2', 'Part II, Item 3', 'Part II, Item 4', 'Part II, Item 5', 'Part II, Item 6']


📄 
├── PART I
│   ├── Item 1   Financial Statements
│   ├── Item 2   Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)
│   ├── Item 3   Quantitative and Qualitative Disclosures About Market Risk
│   └── Item 4   Controls and Procedures
└── PART II
    ├── Item 1   Legal Proceedings
    ├── Item 1A  Risk Factors
    ├── Item 2   Unregistered Sales of Equity Securities and Use of Proceeds
    ├── Item 3   Defaults Upon Senior Securities
    ├── Item 4   Mine Safety Disclosures
    ├── Item 5   Other Information
    └── Item 6   Exhibits

In [36]:
mda = tenq.get_item_with_part("Part I", "Item 2")
risk_factors = tenq.get_item_with_part("Part II", "Item 1A")

(part: str, item: str, markdown: bool = True) -> Optional[str]


In [62]:
class SECIngestor:
    def __init__(self, ticker: str):
        self.ticker = ticker.upper()
        self.company = et.Company(self.ticker)

    def _get_latest_10q(self):
        filings = self.company.get_filings(form="10-Q")
        return filings.latest()

    def _clean_section(self, section):
        if section is None:
            return ""

        lines = [
            line.strip()
            for line in str(section).splitlines()
            if line.strip()
        ]

        return "\n".join(lines)

    def retrieve_filing(self):
        filing = self._get_latest_10q()
        ten_q = filing.obj()

        mda = ten_q.get_item_with_part("Part I", "Item 2")
        risk_factors = ten_q.get_item_with_part(
            "Part II",
            "Item 1A"
        )

        return {
            "ticker": self.ticker,
            "company_name": self.company.name,
            "cik": self.company.cik,
            "filing_type": filing.form,
            "filing_date": filing.filing_date,
            "accession_number": filing.accession_no,
            "source_url": filing.url,
            "sections": {
                "mda": self._clean_section(mda),
                "risk_factors": self._clean_section(risk_factors),
            },
        }

In [68]:
apple_data = SECIngestor("AAPL")
s = apple_data.retrieve_filing()
print(s["cik"])
print(s["sections"]["mda"][:500])
print(s["sections"]["risk_factors"][:500])
print(s["filing_date"])

320193
Item 2.    Management’s Discussion and Analysis of Financial Condition and Results of Operations
This Item and other sections of this Quarterly Report on Form 10-Q (“Form 10-Q”) contain forward-looking statements, within the meaning of the Private Securities Litigation Reform Act of 1995, that involve risks and uncertainties. Forward-looking statements provide current expectations of future events based on certain assumptions and include any statement that does not directly relate to any histori
Item 1A.    Risk Factors
The Company’s business, reputation, results of operations, financial condition and stock price can be materially and adversely affected by a number of factors, whether currently known or unknown, including those described in Part I, Item 1A of the 2025 Form 10-K and Part II, Item 1A of the Form 10-Q for the quarter ended March 28, 2026 (the “second quarter 2026 Form 10-Q”), in each case under the heading “Risk Factors.” Except for the risk factors set forth below